In [3]:
!pip install transformers datasets torch

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader
import torch

In [5]:
# ============================================================
# 1. Load TinyStories in STREAMING mode
# ============================================================
# TinyStories contains short children's stories generated by GPT.
# No config name needed — unlike WikiText-2, it's a single config dataset.
stream_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
)

# Peek at the first example to understand the data structure
first_example = next(iter(stream_dataset))
print("Column names:", list(first_example.keys()))
print("\nSample story:\n", first_example["text"][:300])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Column names: ['text']

Sample story:
 One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and 


In [6]:
# ============================================================
# 2. Filter out short/empty stories
# ============================================================
# Unlike WikiText-2, TinyStories rarely has empty lines,
# but filtering ensures we only keep stories with enough content
# to produce meaningful token chunks.
MIN_TEXT_LENGTH = 100  # characters

filtered_dataset = stream_dataset.filter(
    lambda example: len(example["text"].strip()) >= MIN_TEXT_LENGTH
)

print(f"Filter applied: keeping stories with >= {MIN_TEXT_LENGTH} characters.")

Filter applied: keeping stories with >= 100 characters.


In [7]:
# ============================================================
# 3. Initialize BERT tokenizer
# ============================================================
# bert-base-uncased uses WordPiece tokenization.
# Key differences from GPT-2 BPE:
#   - Vocabulary: ~30k tokens (GPT-2 has ~50k)
#   - Lowercases all text (uncased)
#   - Has a real [PAD] token — no workaround needed
#   - Adds [CLS] and [SEP] special tokens by default (we disable them below)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"Vocab size       : {tokenizer.vocab_size}")
print(f"Pad token        : {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"Special tokens   : {tokenizer.all_special_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size       : 30522
Pad token        : [PAD] (id=0)
Special tokens   : ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']


In [8]:
# ============================================================
# 4. Tokenization step
# ============================================================
# We disable add_special_tokens so [CLS]/[SEP] are not inserted
# between every story — we want a continuous token stream for LM training.
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        add_special_tokens=False  # no [CLS]/[SEP] between chunks
    )

# Lazy tokenization — nothing is processed until we iterate
tokenized_stream = filtered_dataset.map(tokenize_function, batched=True)

# Quick sanity check on one tokenized example
sample = next(iter(tokenized_stream))
print("Token IDs (first 20):", sample["input_ids"][:20])
print("Decoded           :", tokenizer.decode(sample["input_ids"][:20]))

Token indices sequence length is longer than the specified maximum sequence length for this model (948 > 512). Running this sequence through the model will result in indexing errors


Token IDs (first 20): [2028, 2154, 1010, 1037, 2210, 2611, 2315, 7094, 2179, 1037, 12201, 1999, 2014, 2282, 1012, 2016, 2354, 2009, 2001, 3697]
Decoded           : one day, a little girl named lily found a needle in her room. she knew it was difficult


In [9]:
# ============================================================
# 5. Rolling buffer — chunk into fixed-length blocks
# ============================================================
# block_size = 256 (larger than Lab2's 128) because TinyStories
# are longer and more coherent — bigger blocks capture more context.
block_size = 256

def group_texts_streaming(dataset_iter, block_size):
    buffer = []
    for example in dataset_iter:
        buffer.extend(example["input_ids"])
        while len(buffer) >= block_size:
            chunk = buffer[:block_size]
            buffer = buffer[block_size:]
            yield {
                "input_ids": chunk,
                "attention_mask": [1] * block_size
            }

In [10]:
# ============================================================
# 6. Wrap generator in a PyTorch IterableDataset
# ============================================================
class StreamingLMIterableDataset(IterableDataset):
    def __init__(self, hf_iterable_dataset, block_size):
        self.dataset = hf_iterable_dataset
        self.block_size = block_size

    def __iter__(self):
        return group_texts_streaming(self.dataset, self.block_size)

grouped_iterable_dataset = StreamingLMIterableDataset(tokenized_stream, block_size)

In [11]:
# ============================================================
# 7. Collate function
# ============================================================
# attention_mask is meaningful for BERT-style models — we include it
# explicitly as a real tensor (all 1s here since chunks are full blocks).
def collate_fn(batch):
    input_ids = torch.tensor([ex["input_ids"] for ex in batch], dtype=torch.long)
    attention_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone()
    }

In [12]:
# ============================================================
# 8. Create the DataLoader
# ============================================================
train_loader = DataLoader(grouped_iterable_dataset, batch_size=8, collate_fn=collate_fn)

In [13]:
# ============================================================
# 9. Inspect a few batches
# ============================================================
print("Sample streaming batches:")
for i, batch in enumerate(train_loader):
    print(f"Batch {i} -> input_ids: {batch['input_ids'].shape} | attention_mask: {batch['attention_mask'].shape}")
    if i == 2:
        break

# Decode the first chunk from the last batch to verify readable story content
print("\n--- Decoded first chunk of last batch ---")
print(tokenizer.decode(batch["input_ids"][0].tolist()))

Sample streaming batches:
Batch 0 -> input_ids: torch.Size([8, 256]) | attention_mask: torch.Size([8, 256])
Batch 1 -> input_ids: torch.Size([8, 256]) | attention_mask: torch.Size([8, 256])
Batch 2 -> input_ids: torch.Size([8, 256]) | attention_mask: torch.Size([8, 256])

--- Decoded first chunk of last batch ---
saw the present, she was so happy! she gave tim a big hug and thanked him for the special gift. tim felt proud that he could build something so nice for his mom. and he was no longer worried. one day, a big whale was swimming in the deep blue sea. the whale was very mysterious. it had a big smile on its face and liked to play with the little fish. the mysterious whale loved to relax in the warm water. it would lay on its back and let the waves rock it like a baby. the little fish would swim around the whale, and they all felt happy and safe. one day, the mysterious whale and the little fish found a beautiful place to relax. it was a quiet spot with lots of colorful plants and 